For each `c_p` from the grid:

1. **Split.** Each sample is split once (before filtering by `c_p`) by `halfthroat_id` into **tune** and **eval**.
   The split is fixed, so as `c_p` grows the eval sets are nested in each other and the curve over `c_p` is not noisy because of different splits.
2. **Hyperparameter tuning** (optuna) — only on the tune parts, leave-one-sample-out:
   we train on the tune parts of two samples, test on the tune part of the third one, and average $R^2$ over the three samples.
3. **Validation** with the best parameters and this number of trees (without early stopping) — also leave-one-sample-out:
   we train on the *full* (tune + eval) two samples and test on the eval part of the third one.
   The eval parts did not take part in any selection, so this is a fair estimate.
4. `c_p` is selected by the tune-$R^2$, the final quality is the eval-$R^2$ at this `c_p`.

In addition to the usual eval-$R^2$, $R^2$ is computed on the common subset (halfthroats passing the strictest filter `C_P_REF`).
This set is the same for all `c_p`, so it shows whether the filter helps training rather than just removing difficult points from the test.

In [ ]:
import os
import pickle

import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import sklearn.metrics
import matplotlib
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Settings

In [ ]:
DATA_DIR = '../datasets'
SAMPLES = ['Bentheimer', 'Berea', 'Leopard']

TUNE_FRAC = 0.5                      # fraction of halfthroats of each sample used for hyperparameter tuning, the rest is eval
SEED = 42

C_P_GRID = 0.00005 * np.arange(40)
C_P_REF = C_P_GRID.max()             # the strictest filter: common test subset for all c_p

N_TRIALS = 50
NUM_BOOST_ROUND = 1000
EARLY_STOPPING_ROUNDS = 15

DUMP_FILE = f'dump_loso_xgbcv_tune{TUNE_FRAC}.pkl'   # results are appended after each c_p; on restart, finished c_p values are skipped
USE_TEX = False                     # LaTeX in plot labels

## Filtering

In [ ]:
def clean_dataset(data, c_pressure = 0, c_distance = 0, c_volume = 0):
    # a list, not a set: the column order of a set changes between kernel restarts, and XGBoost depends on it
    columns_to_keep = ['halfthroat_id', 'volume', 'g', 'G_vp', 'G_1', 'G_2', 'G_3', 'G_4',
              'border1_relative_area', 'border2_relative_area', 'flow_angle_1', 'flow_angle_2',
              'distance_to_border1_cm', 'distance_to_border2_cm',
              'distance_to_border1_direct', 'distance_to_border2_direct']

    filtered_df = data[data.apply(lambda x: my_custom_criteria(x, c_pressure, c_distance, c_volume), axis=1)]

    filtered_dataset = filtered_df[columns_to_keep].copy()

    clean_data = filtered_dataset.replace([np.inf, -np.inf], np.nan).dropna()

    return clean_data

def my_custom_criteria(row: pd.Series, c_pressure, c_distance, c_volume) -> bool:
    pressure_is_high_enough = min([row["pressure_x"], row["pressure_y"], row["pressure_z"]]) >= c_pressure
    halfthroat_is_far_enough_from_cube_border = row["distance_to_cube_border"] >= c_distance
    connectivty_check = row["G_vp"] > 0
    volume_is_big_enough = row["volume"] >= c_volume

    return (pressure_is_high_enough and
            halfthroat_is_far_enough_from_cube_border and
            connectivty_check and
            volume_is_big_enough)

## Data and the tune / eval split

In [ ]:
data = {name: pd.read_csv(f'{DATA_DIR}/{name}.csv') for name in SAMPLES}

def split_tune_eval(df, tune_frac, seed):
    """Random split of the halfthroat_id values of one sample into tune and eval (before filtering, so it does not depend on c_p)."""
    ids = df['halfthroat_id'].unique()
    np.random.default_rng(seed).shuffle(ids)
    n_tune = int(round(tune_frac * len(ids)))
    return ids[:n_tune], ids[n_tune:]

tune_ids, eval_ids = {}, {}
for name in SAMPLES:
    tune_ids[name], eval_ids[name] = split_tune_eval(data[name], TUNE_FRAC, SEED)

# halfthroats passing the strictest filter form the common test subset for comparing c_p
common_ids = {name: clean_dataset(data[name], c_pressure=C_P_REF)['halfthroat_id'].values for name in SAMPLES}

for name in SAMPLES:
    print(f'{name}: total {len(data[name])}, tune {len(tune_ids[name])}, eval {len(eval_ids[name])}, '
          f'passing C_P_REF {len(common_ids[name])}')

## Leave-one-sample-out and hyperparameter tuning

In [ ]:
def to_Xy(df):
    X = df.drop(columns=['g', 'halfthroat_id', 'volume'])
    y = np.log(df['g'])
    return X, y

def loso_cv(parts):
    """For xgb.cv: one DMatrix from all parts and leave-one-sample-out folds (test is one sample, training is the other two)."""
    df = pd.concat([parts[n] for n in SAMPLES])
    sample = np.concatenate([[n] * len(parts[n]) for n in SAMPLES])
    X, y = to_Xy(df)
    folds = [(np.where(sample != n)[0], np.where(sample == n)[0]) for n in SAMPLES]
    return xgb.DMatrix(X, label=y), folds

def loso_dmatrices(train_parts, test_parts):
    """For validation: for each sample — (name, dtrain from the other train_parts, dtest from test_parts[name])."""
    folds = []
    for test_name in test_parts:
        train = pd.concat([train_parts[n] for n in train_parts if n != test_name])
        X_train, y_train = to_Xy(train)
        X_test, y_test = to_Xy(test_parts[test_name])
        folds.append((test_name, xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_test, label=y_test)))
    return folds

def r2(predt: np.ndarray, dtrain: xgb.DMatrix) -> np.ndarray:
    '''Minus R^2 — the metric for early stopping.'''
    y = dtrain.get_label()
    return 'R2', -sklearn.metrics.r2_score(y, predt)

BASE_PARAMS = {
    'objective': 'reg:squarederror',
    'disable_default_eval_metric': 1,
    'sampling_method': 'uniform',
    'seed': SEED,
}

In [ ]:
def find_best_xgboost_params(dcv, folds, n_trials=N_TRIALS):
    """
    Hyperparameter tuning with xgb.cv, the folds are leave-one-sample-out on the tune parts.
    xgb.cv builds the trees in all folds round by round and stops all folds at the same round by the mean R^2,
    so the number of trees n_rounds is common; in validation it is used without early stopping.
    Returns (params, n_rounds, mean R^2, std R^2 over samples) of the best trial.
    """
    def objective(trial):
        params = {**BASE_PARAMS,
            'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        }

        cv_results = xgb.cv(
            params,
            dcv,
            num_boost_round=NUM_BOOST_ROUND,
            folds=folds,
            custom_metric=r2,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )

        # after early stopping the table is truncated at the best round; r2() returns minus R^2
        trial.set_user_attr('n_rounds', len(cv_results))
        trial.set_user_attr('r2_std', cv_results.iloc[-1]['test-R2-std'])
        return -cv_results.iloc[-1]['test-R2-mean']

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)

    best = study.best_trial
    return {**BASE_PARAMS, **best.params}, best.user_attrs['n_rounds'], best.value, best.user_attrs['r2_std']

## One c_p: tuning on tune, validation on eval

In [ ]:
def get_R2(c_p):
    full, tune, evl = {}, {}, {}
    for name in SAMPLES:
        full[name] = clean_dataset(data[name], c_pressure=c_p)
        tune[name] = full[name][full[name]['halfthroat_id'].isin(tune_ids[name])]
        evl[name] = full[name][full[name]['halfthroat_id'].isin(eval_ids[name])]
    print(f'c_p:{c_p:.5f}, len:', {n: len(full[n]) for n in SAMPLES})

    # 1. hyperparameter tuning: only the tune parts
    params, n_rounds, tune_r2, tune_r2_std = find_best_xgboost_params(*loso_cv(tune))

    # 2. validation: train on the full neighbouring samples, test on the eval part of the held-out sample
    eval_r2, eval_r2_common, predictions = {}, {}, {}
    for test_name, dtrain, dtest in loso_dmatrices(full, evl):
        model = xgb.train(params, dtrain, num_boost_round=n_rounds)
        y_true, y_pred = dtest.get_label(), model.predict(dtest)
        ids = evl[test_name]['halfthroat_id'].values
        common = np.isin(ids, common_ids[test_name])

        eval_r2[test_name] = sklearn.metrics.r2_score(y_true, y_pred)
        eval_r2_common[test_name] = sklearn.metrics.r2_score(y_true[common], y_pred[common])
        predictions[test_name] = pd.DataFrame({'halfthroat_id': ids, 'y_true': y_true, 'y_pred': y_pred})

    print(f'    tune R2 = {tune_r2:.4f} ± {tune_r2_std:.4f}, n_rounds = {n_rounds}')
    print(f'    eval R2 = {np.mean(list(eval_r2.values())):.4f}  {np.round(list(eval_r2.values()), 4)}')

    return {
        'c_p': c_p,
        'n_rows': {n: len(full[n]) for n in SAMPLES},
        'params': params,
        'n_rounds': n_rounds,
        'tune_r2': tune_r2,
        'tune_r2_std': tune_r2_std,
        'eval_r2': eval_r2,
        'eval_r2_common': eval_r2_common,
        'predictions': predictions,
    }

## Run over the c_p grid
The results are saved after each c_p. If you changed `TUNE_FRAC`, `SEED` or the search hyperparameters, delete the old `DUMP_FILE`.

In [ ]:
results = []
if os.path.exists(DUMP_FILE):
    with open(DUMP_FILE, 'rb') as fin:
        results = pickle.load(fin)
done = {round(r['c_p'], 10) for r in results}

for i, c_p in enumerate(C_P_GRID):
    if round(c_p, 10) in done:
        continue
    print(i)
    results.append(get_R2(c_p))
    with open(DUMP_FILE, 'wb') as fout:
        pickle.dump(results, fout)

## Summary and selection of c_p
`c_p` is selected by the tune-$R^2$; the eval-$R^2$ at it is the final estimate.
`eval_R2_std` is the spread between the three samples (not between the folds of a random CV), so it is noticeably larger.

In [ ]:
summary = pd.DataFrame([{
    'c_p': r['c_p'],
    'n_rows': sum(r['n_rows'].values()),
    'n_rounds': r['n_rounds'],
    'tune_R2': r['tune_r2'],
    'tune_R2_std': r['tune_r2_std'],
    'eval_R2': np.mean(list(r['eval_r2'].values())),
    'eval_R2_std': np.std(list(r['eval_r2'].values())),
    'eval_R2_common': np.mean(list(r['eval_r2_common'].values())),
    **{f'eval_R2_{n}': r['eval_r2'][n] for n in SAMPLES},
} for r in results]).sort_values('c_p').reset_index(drop=True)

best = summary.loc[summary['tune_R2'].idxmax()]
best_result = next(r for r in results if np.isclose(r['c_p'], best['c_p']))

print(summary.round(4).to_string())
print()
print(f"Best c_p (by tune): {best['c_p']:.5f}")
print(f"eval R2 = {best['eval_R2']:.4f} ± {best['eval_R2_std']:.4f}   " +
      ', '.join(f"{n}: {best[f'eval_R2_{n}']:.4f}" for n in SAMPLES))
print('Parameters:', best_result['params'], ' n_rounds:', best_result['n_rounds'])

## R² vs c_p

In [ ]:
ax_fontsize = 12

matplotlib.style.use('default')
matplotlib.rcParams['font.family'] = "serif"
matplotlib.rcParams['mathtext.fontset'] = "dejavuserif"
matplotlib.rcParams['text.usetex'] = USE_TEX
matplotlib.rcParams['text.latex.preamble'] = r"\usepackage{amsmath} \usepackage{amssymb}"

fig, ax = plt.subplots(figsize=(2.7, 2), layout = "constrained")

ax.plot(summary['c_p'], summary['eval_R2'], zorder=2, label='eval', color='black')
ax.plot(summary['c_p'], summary['tune_R2'], '--', zorder=2, label='tune', color='tab:red')

for n in SAMPLES:
    ax.plot(summary['c_p'], summary[f'eval_R2_{n}'], label=n, lw=0.8)
ax.axvline(best['c_p'], color='grey', lw=0.8, zorder=1)
ax.set_xlabel(r'$\Delta p$ criterion', fontsize=ax_fontsize)
ax.set_ylabel(r'$R^2$', fontsize=ax_fontsize)
ax.set_ylim(bottom=0.56)
ax.legend(fontsize=8)

ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))

plt.savefig('R2_loso.pdf')

## Final plot at the selected c_p
Predictions are the eval parts of all three samples, each predicted by a model trained on the other two samples.

In [ ]:
pred = pd.concat(best_result['predictions'].values())
y_exp, y_pred = pred['y_true'].values, pred['y_pred'].values

ax_fontsize = 14

fig, axs = plt.subplot_mosaic([['a)', 'b)']],
                              layout='constrained', figsize=(6.5, 3))

axs['a)'].plot(np.exp([-100, 6]), np.exp([-100, 6]), alpha=0.3, zorder = 2, color = 'black')
axs['a)'].set_ylabel(r'$g_{predicted}$', fontsize=ax_fontsize)
axs['a)'].set_xlabel(r'$g_{actual}$', fontsize=ax_fontsize)

axs['a)'].set_xlim(np.exp([-11, -1]))
axs['a)'].set_ylim(np.exp([-11, -1]))

axs['a)'].hexbin(np.exp(y_exp), np.exp(y_pred), gridsize=150, bins='log', xscale = 'log', yscale = 'log', cmap='Blues', mincnt=1, zorder=1, linewidths=0.1)

axs['a)'].text(0.00003, 0.1, rf"$R^2 = {best['eval_R2']:.3f} \pm {best['eval_R2_std']:.3f}$", fontsize = ax_fontsize)

axs['b)'].hist(y_pred - y_exp, bins = 30, range=(-3, 3), zorder = 2, edgecolor = 'black', color = '#8fbbd9')

axs['b)'].set_xlabel(r'$\ln{g_{predicted} / g_{actual}}$', fontsize=ax_fontsize)

axs['b)'].grid(axis='y', zorder=0)

axs['a)'].set_title('a)', loc='left', fontsize=ax_fontsize)
axs['b)'].set_title('b)', loc='left', fontsize=ax_fontsize)

plt.savefig('final_loso.pdf')